In [5]:
from newspaper import Article, Source
from newspaper import build
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

# ===== ENTER YOUR PAGE URL HERE =====
PAGE_URL = "https://www.thehindu.com/todays-paper/2026-01-29/th_chennai/"


def get_article_links(page_url: str) -> list:
    """Get all article links from a page using requests + BeautifulSoup."""
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
    
    try:
        resp = requests.get(page_url, headers=headers, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"Error: {e}")
        return []
    
    soup = BeautifulSoup(resp.text, "html.parser")
    base_domain = urlparse(page_url).netloc
    
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        full_url = urljoin(page_url, href)
        parsed = urlparse(full_url)
        
        # Only article links from same domain
        if parsed.netloc.endswith(base_domain.replace('www.', '')) and '/article' in full_url:
            links.add(full_url)
    
    return sorted(links)


def fetch_article(url: str) -> dict:
    """Fetch and parse a single article using newspaper3k."""
    try:
        article = Article(url)
        article.download()
        article.parse()
        
        return {
            'url': url,
            'title': article.title,
            'authors': article.authors,
            'publish_date': str(article.publish_date) if article.publish_date else None,
            'text': article.text,
            'summary': article.text[:500] + '...' if len(article.text) > 500 else article.text,
            'top_image': article.top_image,
            'success': True
        }
    except Exception as e:
        return {'url': url, 'error': str(e), 'success': False}


# Get all article links
print(f"📰 Fetching links from: {PAGE_URL}")
article_links = get_article_links(PAGE_URL)
print(f"🔗 Found {len(article_links)} article links\n")

# Display the links
for i, link in enumerate(article_links, 1):
    print(f"{i:2}. {link}")

📰 Fetching links from: https://www.thehindu.com/todays-paper/2026-01-29/th_chennai/
🔗 Found 45 article links

 1. https://frontline.thehindu.com/politics/arvind-kejriwal-exclusive-interview-delhi-election-2025-voting-aap-vs-bjp-vs-congress/article69131147.ece?utm_source=th&utm_medium=footer&utm_campaign=internal
 2. https://frontline.thehindu.com/the-nation/waqf-amendment-bill-impact-indian-muslims-reforms-consultative-process-othering-backwardness/article68849163.ece?utm_source=th&utm_medium=footer&utm_campaign=internal
 3. https://sportstar.thehindu.com/cricket/india-vs-england-test-series-result-eng-vs-ind-full-list-wins-results-tests/article69704557.ece
 4. https://sportstar.thehindu.com/cricket/ipl/ipl-news/ipl-auction-2026-live-updates-csk-kkr-rcb-mi-srh-dc-pbks-lsg-gt-rr-team-list-sold-unsold-players/article70401822.ece
 5. https://www.thehindu.com/books/finding-his-mark-tullys-books-chronicled-india-on-the-move/article70561080.ece
 6. https://www.thehindu.com/business/budget/ec

In [6]:
# Fetch all articles with full content using newspaper3k
print(f"📖 Fetching {len(article_links)} articles...\n")

articles = []
for i, url in enumerate(article_links, 1):
    print(f"[{i}/{len(article_links)}] Fetching: {url[:70]}...")
    article_data = fetch_article(url)
    articles.append(article_data)
    
    if article_data['success']:
        print(f"    ✅ {article_data['title'][:60]}...")
    else:
        print(f"    ❌ Error: {article_data.get('error', 'Unknown')}")

# Summary
successful = [a for a in articles if a['success']]
print(f"\n{'='*70}")
print(f"✅ Successfully fetched: {len(successful)}/{len(articles)} articles")

📖 Fetching 45 articles...

[1/45] Fetching: https://frontline.thehindu.com/politics/arvind-kejriwal-exclusive-inte...
    ✅ Arvind Kejriwal Exclusive Interview | ‘AAP will form the gov...
[2/45] Fetching: https://frontline.thehindu.com/the-nation/waqf-amendment-bill-impact-i...
    ✅ Editor’s Note | Vaishna Roy Writes: Why the Waqf Bill Reform...
[3/45] Fetching: https://sportstar.thehindu.com/cricket/india-vs-england-test-series-re...
    ✅ IND vs ENG Tests in England: Full list of England vs India T...
[4/45] Fetching: https://sportstar.thehindu.com/cricket/ipl/ipl-news/ipl-auction-2026-l...
    ✅ Editor’s Note | Vaishna Roy Writes: Why the Waqf Bill Reform...
[3/45] Fetching: https://sportstar.thehindu.com/cricket/india-vs-england-test-series-re...
    ✅ IND vs ENG Tests in England: Full list of England vs India T...
[4/45] Fetching: https://sportstar.thehindu.com/cricket/ipl/ipl-news/ipl-auction-2026-l...
    ✅ IPL Auction 2026 HIGHLIGHTS: Cameron Green signs for KKR for...
[5/45] 

In [7]:
# Display all fetched articles with their content
print("📰 FETCHED ARTICLES\n" + "="*70)

for i, article in enumerate(successful, 1):
    print(f"\n{'─'*70}")
    print(f"📌 Article {i}: {article['title']}")
    print(f"🔗 {article['url']}")
    if article['authors']:
        print(f"✍️  Authors: {', '.join(article['authors'])}")
    if article['publish_date']:
        print(f"📅 Date: {article['publish_date']}")
    print(f"\n📝 Content Preview:\n{article['summary']}")

📰 FETCHED ARTICLES

──────────────────────────────────────────────────────────────────────
📌 Article 1: Arvind Kejriwal Exclusive Interview | ‘AAP will form the government again in Delhi with a comfortable majority’
🔗 https://frontline.thehindu.com/politics/arvind-kejriwal-exclusive-interview-delhi-election-2025-voting-aap-vs-bjp-vs-congress/article69131147.ece?utm_source=th&utm_medium=footer&utm_campaign=internal
📅 Date: 2025-01-25 11:20:45+00:00

📝 Content Preview:
Published : Jan 25, 2025 16:50 IST - 17 MINS READ

It has been difficult to get an interview with Arvind Kejriwal. Not only is it a question of time, Kejriwal is also wary of “mainstream” media, a fact that he admits when we finally meet. After a postponement, the interview is set for 9:00 pm, and when we get to his current residence at 5 Ferozeshah Road, Kejriwal has just returned from a long day of campaigning for what is being seen as a make-or-break election for both him and his party.

But ...

───────────────────────